In [ ]:
"""
After manual analysis, I've found some mistakes happening in the DB results.
In this notebook, I pick the incorrect records to test the DB_1record_evaluator, seeing if it successfully tells that these records are incorrect.
Extraction of text_doc and file_path_doc is also done here, so that it can be done once for 1 notegroup instead of repeatedly done for each record
"""
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from oral_notes.evaluate.DB_1record_evaluator import DB1recordEvaluator
from utils.html_viewer import show
import sqlite3

pipeline_type="baseline_v2"
DB_PATH = "DB/oedb_baseline_v2.db"
schema_path = "data/metadata_DB/schema_v2.yaml"
prompt_path_evaluator = "data/prompt_templates/prompt_evaluator.yaml"
service_account_file="config/service_account_key.json"

def textdoc_extractor(notegroup_id, DB_PATH, service_account_file):
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()
        if row is None:
            raise ValueError(f"No notegroup found with ID {notegroup_id}")
        note_url_qa, note_url_participant = row

    file_loader = GoogleDriveLoader(service_account_file)
    extractor = TextExtractor()

    all_texts = {}
    all_drive_paths = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            print(f"\n--- Skipping {label}: no URL ---")
            continue
        print(f"\n--- Loading and extracting text from {label} ---")
        result = file_loader.load(url)
        text = extractor.extract(result)
        all_texts[label] = f"[Data source: {result['name']}]\n{text}"
        all_drive_paths[label] = result['drive_path']
    combined_drive_paths = "|".join(all_drive_paths.values())
    combined_text = "\n---\n".join(
        t for t in [all_texts.get('PARTICIPANT', ''), all_texts.get('QA', '')] if t
    )
    return combined_text, combined_drive_paths

In [ ]:
evaluator_version="initial_test"
notegroupID=3
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
# run the evaluator over the target records
records_to_evaluate = (
    [("notegroups", 3)]
    + [("participants", pk) for pk in range(14, 19)]   # 14-18
    + [("questions", pk) for pk in range(44, 49)]      # 44-48
)

for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")